<a href="https://colab.research.google.com/github/Dukeman7/MARCO_LEGAL_TELECOM_LD/blob/main/ANALIZADOR_DE_DESPLIEGUES_THUNDER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. CONEXIÓN A DRIVE
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/DATA/THUNDERNET'

def procesar_hoja_posicional(path, palabras_clave, col_nombre):
    """Limpia el Excel (Salta F1 y C1) y extrae por posición real"""
    try:
        xl = pd.ExcelFile(path)
        hoja = [h for h in xl.sheet_names if any(p in h.upper() for p in palabras_clave)]
        if not hoja: return pd.DataFrame()

        # REGLA JUANCHO: Salta Fila 1 (skiprows=1 hace que la Fila 2 sea el Header)
        df = pd.read_excel(path, sheet_name=hoja[0], skiprows=1)

        # REGLA JUANCHO: Salta Columna 1 (La columna 'A' de Excel es el índice 0 en Python)
        df = df.iloc[:, 1:]

        # --- EXTRACCIÓN POR COORDENADAS (Hoja de Ruta) ---
        # 1: MES-AÑO | 4: ESTADO | 5: MUNICIPIO | 6: VALOR (Última relevante)
        temp = pd.DataFrame()
        temp['MES-AÑO'] = df.iloc[:, 1].astype(str).str.strip().str.upper()
        temp['ESTADO'] = df.iloc[:, 4].astype(str).str.strip().str.upper()
        temp['MUNICIPIO'] = df.iloc[:, 5].astype(str).str.strip().str.upper()
        temp[col_nombre] = pd.to_numeric(df.iloc[:, 6], errors='coerce').fillna(0)

        # SUMA DE REGISTROS DIARIOS: Convertimos el log diario en un total mensual
        return temp.groupby(['MES-AÑO', 'ESTADO', 'MUNICIPIO'])[col_nombre].sum().reset_index()
    except Exception as e:
        return pd.DataFrame()

# --- PROCESO DE LOS 15 MONSTRUOS ---
archivos = [f for f in os.listdir(folder_path) if f.endswith('.xlsx') and not f.startswith('~$')]
lista_final = []

for arc in archivos:
    p = os.path.join(folder_path, arc)
    print(f"🛠️ Convirtiendo y Sincerando: {arc}")

    df_a = procesar_hoja_posicional(p, ['ABONADO'], 'ABONADOS')
    df_t = procesar_hoja_posicional(p, ['TRONCAL', 'TRANSPORTE'], 'TRONCAL')
    df_m = procesar_hoja_posicional(p, ['MILLA'], 'MILLA')

    if not df_a.empty:
        # Unimos las 3 dimensiones en una sola tabla mensual
        merged = df_a
        if not df_t.empty: merged = pd.merge(merged, df_t, on=['MES-AÑO', 'ESTADO', 'MUNICIPIO'], how='outer')
        if not df_m.empty: merged = pd.merge(merged, df_m, on=['MES-AÑO', 'ESTADO', 'MUNICIPIO'], how='outer')
        lista_final.append(merged)

if lista_final:
    # EL CHORIZO CRUDO PERO LIMPIO
    df_master = pd.concat(lista_final, ignore_index=True).fillna(0)

    # Limpieza final de filas de "TOTAL" o basura de pie de página
    df_master = df_master[~df_master['ESTADO'].str.contains('TOTAL|NAN|ESTADO|UNNAMED', na=False)]

    df_master.to_csv('THUNDERNET_15_MESES_LIMPIO.csv', index=False)
    print("\n🔥 ¡DOMADOS! Archivo 'THUNDERNET_15_MESES_LIMPIO.csv' generado.")
    display(df_master.head(15))